In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
import os
import sklearn

In [2]:
sklearn.set_config(transform_output="pandas")

In [3]:
# Load data
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [4]:
X = train.drop(['id', 'addicted_label'], axis=1)
y = train['addicted_label']
X_test = test.drop(['id'], axis=1)

In [5]:
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['number']).columns.tolist()

/tmp/ipykernel_10675/3837567942.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()


In [6]:
# 1. Imputation
imputer = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numerical_cols),
        ('cat', SimpleImputer(strategy='most_frequent'), categorical_cols)
    ],
    verbose_feature_names_out=False
)

In [7]:
# 2. Feature Engineering
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_out = X.copy()
        
        denom_screen = X_out['daily_screen_time_hours'].replace(0, 0.001)
        denom_notif = X_out['notifications_per_day'].replace(0, 0.001)
        
        X_out['social_media_ratio'] = X_out['social_media_hours'] / denom_screen
        X_out['gaming_ratio'] = X_out['gaming_hours'] / denom_screen
        X_out['work_study_ratio'] = X_out['work_study_hours'] / denom_screen
        
        X_out['app_opens_per_hour'] = X_out['app_opens_per_day'] / denom_screen
        X_out['notifications_to_opens_ratio'] = X_out['app_opens_per_day'] / denom_notif
        X_out['sleep_deficit'] = 8.0 - X_out['sleep_hours']
        
        return X_out

In [8]:
# 3. Final Preprocessing
final_preprocessor = ColumnTransformer(
    transformers=[
        ('cat_encode', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

In [9]:
preprocessor = Pipeline(steps=[
    ('imputer', imputer),
    ('engineer', FeatureEngineer()),
    ('final_preprocessor', final_preprocessor)
])

In [10]:
print("Preprocessing data...")
X_preprocessed = preprocessor.fit_transform(X)
X_test_preprocessed = preprocessor.transform(X_test)

Preprocessing data...


In [11]:
for col in categorical_cols:
    X_preprocessed[col] = X_preprocessed[col].astype(int).astype('category')
    X_test_preprocessed[col] = X_test_preprocessed[col].astype(int).astype('category')

In [12]:
print("Training base models...")

Training base models...


In [13]:
# Define Base Models
lgbm = LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1)
# For XGBoost, category dtype needs enable_categorical=True
xgb = XGBClassifier(n_estimators=200, learning_rate=0.05, random_state=42, enable_categorical=True, tree_method='hist')
# For CatBoost, pass categorical features
cat_features_indices = [X_preprocessed.columns.get_loc(col) for col in categorical_cols]
cat = CatBoostClassifier(iterations=200, learning_rate=0.05, random_seed=42, verbose=0, cat_features=cat_features_indices)

In [14]:
# Fit models on full train data for simplicity in blending
print("Fitting LightGBM...")
lgbm.fit(X_preprocessed, y)
print("Fitting XGBoost...")
xgb.fit(X_preprocessed, y)
print("Fitting CatBoost...")
cat.fit(X_preprocessed, y)

Fitting LightGBM...


Fitting XGBoost...


Fitting CatBoost...


CatBoostClassifier(cat_features=[0, 1, 2], iterations=200, learning_rate=0.05, random_seed=42, verbose=0)

In [15]:
# Predict on test
print("Predicting on test data...")
preds_lgbm = lgbm.predict_proba(X_test_preprocessed)[:, 1]
preds_xgb = xgb.predict_proba(X_test_preprocessed)[:, 1]
preds_cat = cat.predict_proba(X_test_preprocessed)[:, 1]

Predicting on test data...


In [16]:
# Simple Blending (Average)
preds_blend = (preds_lgbm + preds_xgb + preds_cat) / 3.0

In [17]:
os.makedirs('submissions', exist_ok=True)
submission = pd.DataFrame({'id': test['id'], 'addicted_label': preds_blend})
submission.to_csv('submissions/ensembling.csv', index=False)
print("Submission saved to submissions/ensembling.csv")

Submission saved to submissions/ensembling.csv
